In [0]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
w.repos.update(w.workspace.get_status(path="/Users/anish9167473766@gmail.com/project_voltkart_codebasics.io").object_id, branch="branch1")

In [0]:
import requests


def get_api_data(url, timeout=10):
    """Fetch data from an API and return the extracted JSON payload.

    Args:
        url (str): The API endpoint to call.
        timeout (int): Seconds to wait for the request before failing (default: 10).

    Returns:
        dict: Parsed JSON from the response, or None if the request failed.
    """
    try:
        response = requests.get(url, timeout=timeout)
        response.raise_for_status()  # Raise an error for 4xx/5xx status codes
        return response.json()
    except requests.exceptions.Timeout:
        print(f"Request to {url} timed out after {timeout} seconds.")
        return None
    except requests.exceptions.RequestException as e:
        print(f"Request to {url} failed: {e}")
        return None


# Extract products from the API with a 10 second timeout
products_data = get_api_data("https://dummyjson.com/products/", timeout=10)

if products_data:
    products = products_data.get("products", [])
    print(f"Retrieved {len(products)} products (total: {products_data.get('total', 'unknown')})")
    for product in products[:5]:
        print(f"- {product.get('title')} (${product.get('price')})")


In [0]:
%pip install yfinance

In [0]:
import yfinance as yf


def get_stock_details(symbol, period="1mo"):
    """Fetch daily OHLCV data for a single stock from Yahoo Finance.

    Args:
        symbol (str): Yahoo Finance ticker, e.g. "HDFCBANK.NS" (NSE) or "HDFCBANK.BO" (BSE).
        period (str): History window - "5d", "1mo", "6mo", "1y", "5y", "max".

    Returns:
        DataFrame: Spark DataFrame with Date, symbol, open, high, low, close, volume.
    """
    pdf = yf.download(symbol, period=period, auto_adjust=True, progress=False)
    pdf = pdf.reset_index()
    pdf.columns = pdf.columns.get_level_values(0)  # flatten MultiIndex (Price, Ticker) -> Price
    pdf.columns = [str(c).lower() for c in pdf.columns]  # Open -> open, etc.
    pdf["symbol"] = symbol
    df = spark.createDataFrame(pdf)
    return df.withColumn("date", df["date"].cast("date"))


hdfc_df = get_stock_details("HDFCBANK.NS", period="1mo")
display(hdfc_df.orderBy("date"))
print(f"HDFCBANK.NS: {hdfc_df.count()} trading days")

In [0]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# ---- Build the dataset: each row = day t-1's OHLCV -> day t's close ----
data_pdf = get_stock_details("HDFCBANK.NS", period="2y").orderBy("date").toPandas()

lagged = data_pdf[["open", "high", "low", "close", "volume"]].shift(1)
lagged.columns = ["prev_open", "prev_high", "prev_low", "prev_close", "prev_volume"]
data = pd.concat([lagged, data_pdf["close"].rename("target_close")], axis=1).dropna().reset_index(drop=True)

features = ["prev_open", "prev_high", "prev_low", "prev_close", "prev_volume"]
X, y = data[features], data["target_close"]

# ---- Temporal split: train on the earlier 80%, test on the most recent 20% ----
split = int(len(data) * 0.8)
X_train, X_test, y_train, y_test = X.iloc[:split], X.iloc[split:], y.iloc[:split], y.iloc[split:]

model = LinearRegression().fit(X_train, y_train)
holdout_r2 = r2_score(y_test, model.predict(X_test))
naive_r2 = r2_score(y_test, X_test["prev_close"])  # baseline: predict tomorrow = today

# ---- Influence: standardized coefficients (comparable across different scales) ----
std_coef = model.coef_ * X_train.std() / y_train.std()
influence = pd.DataFrame(
    {"prev_day_feature": features, "std_coefficient": std_coef}
).reindex(std_coef.abs().sort_values(ascending=False).index)
display(influence)

# ---- Predict the next trading day's close from the latest day's OHLCV ----
last_row = data_pdf[["open", "high", "low", "close", "volume"]].tail(1)
last_row.columns = features  # match the prev_* names used during fit
pred_next_close = model.predict(last_row)[0]
print(f"Samples: train={len(X_train)}, test={len(X_test)}")
print(f"Holdout R2 (model): {holdout_r2:.4f} | baseline tomorrow=today R2: {naive_r2:.4f}")
print(f"Predicted next-day close: {pred_next_close:.2f} INR (last actual close: {data_pdf['close'].iloc[-1]:.2f})")

In [0]:
#
# High-conviction buy/sell algorithm.
# Principle: fewer calls - act ONLY when very sure.
#   Entry: full confluence (short SMA > long SMA AND RSI in a healthy momentum
#          band) must hold for K consecutive days, all lagged.
#   Exit:  ONLY a genuine trend break (SMA cross down) or extreme overbought
#          (RSI > 85). No flip-flopping on day-to-day wiggles.
# Tuned on the earlier 70% of history, judged once on the held-out last 30%.
# All indicators are lagged (shift(1)): day t only sees data up to t-1.
#
import numpy as np
import pandas as pd

px = get_stock_details("HDFCBANK.NS", period="5y").orderBy("date").toPandas().reset_index(drop=True)
c = px["close"]
next_ret = c.pct_change().shift(-1)  # return from day t's close to day t+1's close

delta = c.diff()
gain = delta.clip(lower=0).rolling(14).mean()
loss = (-delta.clip(upper=0)).rolling(14).mean()
rsi = (100 - 100 / (1 + gain / loss)).shift(1)


def backtest(short, long_w, r_lo, r_hi, k, idx):
    """Sticky high-conviction strategy: once long, stay long until a real exit."""
    sma_s = c.rolling(short).mean().shift(1)
    sma_l = c.rolling(long_w).mean().shift(1)
    cond = (sma_s > sma_l) & (rsi > r_lo) & (rsi < r_hi)
    confirmed = cond.rolling(k).sum() == k  # K consecutive days of full agreement
    exit_sig = (sma_s < sma_l) | (rsi > 85)
    conf_v = confirmed.fillna(False).values
    exit_v = exit_sig.fillna(True).values
    pos = np.zeros(len(c))
    for i in range(1, len(c)):  # stateful walk forward
        if pos[i - 1] == 1:
            pos[i] = 0.0 if exit_v[i] else 1.0
        elif conf_v[i]:
            pos[i] = 1.0
    pos = pd.Series(pos, index=c.index)
    r = (pos * next_ret).loc[idx].fillna(0)
    total = (1 + r).prod() - 1
    sharpe = r.mean() / (r.std() + 1e-12) * np.sqrt(252)
    return pos, total, sharpe


# Chronological split: tune on the first 70%, judge on the last 30%
idx = px.index[200:]  # skip the 200-day SMA warmup
split = int(len(idx) * 0.7)
idx_train, idx_test = idx[:split], idx[split:]

grid = [
    (s, lw, rlo, rhi, k)
    for s, lw in ((20, 100), (50, 100), (50, 200))
    for rlo, rhi in ((30, 70), (40, 70), (40, 80))
    for k in (3, 5)
]
results = []
for s, lw, rlo, rhi, k in grid:
    _, tr_tot, tr_sh = backtest(s, lw, rlo, rhi, k, idx_train)
    results.append({"short": s, "long": lw, "rsi_lo": rlo, "rsi_hi": rhi, "k": k, "train_ret": tr_tot, "train_sharpe": tr_sh})
sweep = pd.DataFrame(results).sort_values("train_sharpe", ascending=False)
best = sweep.iloc[0]
S, L, RLO, RHI, K = (int(best[col]) for col in ("short", "long", "rsi_lo", "rsi_hi", "k"))

# ---- Winner's calls and out-of-sample evaluation ----
_, tr_tot, tr_sh = backtest(S, L, RLO, RHI, K, idx_train)
pos, te_tot, te_sh = backtest(S, L, RLO, RHI, K, idx_test)

sig = px.loc[idx].copy()
sig["sma_s"] = c.rolling(S).mean().shift(1).loc[idx]
sig["sma_l"] = c.rolling(L).mean().shift(1).loc[idx]
sig["rsi_14"] = rsi.loc[idx]
sig["position"] = pos.loc[idx]
sig["trade"] = sig["position"].diff().fillna(0)

bh_tr = c.loc[idx_train[-1]] / c.loc[idx_train[0]] - 1
bh_te = c.loc[idx_test[-1]] / c.loc[idx_test[0]] - 1
win_rate = (next_ret.loc[sig.index][sig["position"] == 1] > 0).mean()

calls = sig.loc[sig["trade"] != 0, ["date", "close", "sma_s", "sma_l", "rsi_14"]].assign(
    call=np.where(sig.loc[sig["trade"] != 0, "trade"] > 0, "BUY", "SELL")
)
display(calls)

last = sig.iloc[-1]
today_call = "BUY / stay long" if last["position"] == 1 else "STAND ASIDE (no high-conviction signal)"
entries = int((sig["trade"] > 0).sum())
print(f"Best params: SMA {S}/{L}, RSI band {RLO}-{RHI}, {K}-day confirmation ({len(grid)} combos tried)")
print(f"TRAIN : strategy {tr_tot:.1%} (Sharpe {tr_sh:.2f}) vs buy & hold {bh_tr:.1%}")
print(f"TEST  : strategy {te_tot:.1%} (Sharpe {te_sh:.2f}) vs buy & hold {bh_te:.1%}")
print(f"Entries in 5y: {entries} | win rate while long: {win_rate:.1%}")
print(f"Today's call ({last['date']}, close {last['close']:.2f}): {today_call} | RSI: {last['rsi_14']:.0f}")

In [0]:
#
# Watchlist scan: run the high-conviction algorithm across NSE stocks.
# Same discipline as before (trend + RSI band + K-day confirmation, sticky exits,
# all indicators lagged). Params are tuned per stock on the first 70% of its own
# history and judged on the held-out last 30%.
#
import time

import numpy as np
import pandas as pd
import yfinance as yf

# NIFTY 50 constituents (verified against Yahoo Finance 2026-11-21; LTIM.NS returns
# no data and is excluded). Symbols with no/short data are auto-excluded below.
WATCHLIST = [
    "ADANIENT.NS", "ADANIPORTS.NS", "APOLLOHOSP.NS", "ASIANPAINT.NS", "AXISBANK.NS",
    "BAJAJ-AUTO.NS", "BAJFINANCE.NS", "BAJAJFINSV.NS", "BEL.NS", "BHARTIARTL.NS",
    "CIPLA.NS", "COALINDIA.NS", "DRREDDY.NS", "EICHERMOT.NS", "ETERNAL.NS",
    "GRASIM.NS", "HCLTECH.NS", "HDFCBANK.NS", "HDFCLIFE.NS", "HEROMOTOCO.NS",
    "HINDALCO.NS", "HINDUNILVR.NS", "ICICIBANK.NS", "INDIGO.NS", "INDUSINDBK.NS",
    "INFY.NS", "ITC.NS", "JIOFIN.NS", "JSWSTEEL.NS", "KOTAKBANK.NS",
    "LT.NS", "M&M.NS", "MARUTI.NS", "NTPC.NS", "ONGC.NS",
    "POWERGRID.NS", "RELIANCE.NS", "SBILIFE.NS", "SBIN.NS", "SHRIRAMFIN.NS",
    "SUNPHARMA.NS", "TATACONSUM.NS", "TCS.NS", "TECHM.NS", "TITAN.NS",
    "TMPV.NS", "TRENT.NS", "ULTRACEMCO.NS", "WIPRO.NS",
]

GRID = [
    (s, lw, rlo, rhi, k)
    for s, lw in ((20, 100), (50, 100), (50, 200))
    for rlo, rhi in ((30, 70), (40, 70), (40, 80))
    for k in (3, 5)
]


def prep(symbol):
    """Fetch 5y daily OHLCV + lagged indicators. Returns None if no data comes back."""
    pdf = yf.download(symbol, period="5y", auto_adjust=True, progress=False)
    if pdf.empty:  # transient failure (e.g. rate limit) - wait and retry once
        time.sleep(3)
        pdf = yf.download(symbol, period="5y", auto_adjust=True, progress=False)
    if pdf.empty:
        return None
    pdf = pdf.reset_index()
    pdf.columns = pdf.columns.get_level_values(0)  # flatten (Price, Ticker) MultiIndex
    pdf.columns = [str(col).lower() for col in pdf.columns]
    c = pdf["close"]
    next_ret = c.pct_change().shift(-1)
    delta = c.diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = (-delta.clip(upper=0)).rolling(14).mean()
    rsi = (100 - 100 / (1 + gain / loss)).shift(1)
    return pdf, c, next_ret, rsi


def conviction_position(c, rsi, short, long_w, r_lo, r_hi, k):
    """Sticky high-conviction position: enter after k confirmed days, exit only on trend break or extreme overbought."""
    sma_s = c.rolling(short).mean().shift(1)
    sma_l = c.rolling(long_w).mean().shift(1)
    cond = (sma_s > sma_l) & (rsi > r_lo) & (rsi < r_hi)
    confirmed = cond.rolling(k).sum() == k
    exit_sig = (sma_s < sma_l) | (rsi > 85)
    conf_v = confirmed.fillna(False).values
    exit_v = exit_sig.fillna(True).values
    pos = np.zeros(len(c))
    for i in range(1, len(c)):
        if pos[i - 1] == 1:
            pos[i] = 0.0 if exit_v[i] else 1.0
        elif conf_v[i]:
            pos[i] = 1.0
    return pd.Series(pos, index=c.index)


def perf(pos, next_ret, idx):
    r = (pos * next_ret).loc[idx].fillna(0)
    total = (1 + r).prod() - 1
    sharpe = r.mean() / (r.std() + 1e-12) * np.sqrt(252)
    return total, sharpe


summary, todays = [], []
for sym in WATCHLIST:
    got = prep(sym)
    if got is None:
        print(f"Skipped {sym}: no data returned")
        continue
    px, c, next_ret, rsi = got
    time.sleep(1)  # stay polite with the data source between symbols
    if len(px) < 300:  # too little history for the 200-day SMA warmup + backtest split
        print(f"Skipped {sym}: insufficient history ({len(px)} rows)")
        continue
    idx = px.index[200:]
    split = int(len(idx) * 0.7)
    idx_train, idx_test = idx[:split], idx[split:]

    # tune on this stock's train window
    best_sh, best_p = -9.0, None
    for s, lw, rlo, rhi, k in GRID:
        pos = conviction_position(c, rsi, s, lw, rlo, rhi, k)
        _, sh = perf(pos, next_ret, idx_train)
        if sh > best_sh:
            best_sh, best_p = sh, (s, lw, rlo, rhi, k)

    pos = conviction_position(c, rsi, *best_p)
    tr_tot, tr_sh = perf(pos, next_ret, idx_train)
    te_tot, te_sh = perf(pos, next_ret, idx_test)
    bh_tr = c.loc[idx_train[-1]] / c.loc[idx_train[0]] - 1
    bh_te = c.loc[idx_test[-1]] / c.loc[idx_test[0]] - 1
    entries = int((pos.loc[idx].diff().fillna(0) > 0).sum())

    last_pos = int(pos.iloc[-1])
    prev_pos = int(pos.iloc[-2])
    last_rsi = rsi.iloc[-1]
    # Entry/exit points: a position change on the latest day is an actionable signal,
    # tradable at today's close (indicators only use data up to yesterday).
    if last_pos == 1 and prev_pos == 0:
        call = "BUY now"  # conviction entry fired today - entry point = today's close
    elif last_pos == 0 and prev_pos == 1:
        call = "SELL now"  # exit fired today (trend break / extreme overbought)
    elif last_pos == 1:
        call = "HOLD (long)"  # already in position, no new signal
    else:
        call = "STAND ASIDE"  # flat, no high-conviction setup

    summary.append({
        "symbol": sym, "best_params": f"SMA {best_p[0]}/{best_p[1]}, RSI {best_p[2]}-{best_p[3]}, k={best_p[4]}",
        "train_ret": tr_tot, "train_vs_bh": tr_tot - bh_tr,
        "test_ret": te_tot, "test_vs_bh": te_tot - bh_te,
        "entries_5y": entries,
    })
    todays.append({"symbol": sym, "close": c.iloc[-1], "rsi_14": last_rsi, "today_call": call})

summary_df = pd.DataFrame(summary)
todays_df = pd.DataFrame(todays)
display(summary_df)
display(todays_df)

active = todays_df[todays_df["today_call"].isin(["BUY now", "HOLD (long)"])]["symbol"].tolist()
buys_now = todays_df.loc[todays_df["today_call"] == "BUY now", "symbol"].tolist()
sells_now = todays_df.loc[todays_df["today_call"] == "SELL now", "symbol"].tolist()
print(f"Scanned {len(summary)} stocks")
print(f"BUY now (fresh entries): {buys_now if buys_now else 'none'}")
print(f"SELL now (fresh exits): {sells_now if sells_now else 'none'}")
print(f"Long right now (incl. fresh buys): {active if active else 'none'}")
print(f"Median outperformance vs buy & hold - train: {summary_df['train_vs_bh'].median():.1%}, test: {summary_df['test_vs_bh'].median():.1%}")